# 📱 Mobile Money Data Extractor — v3
**CSC 3221 — Introduction to Data Science | ICT University**

### What's new in v3
- **Interactive owner confirmation** — auto-detection is shown before processing; you confirm, correct, or skip each file
- **Mojibake fix** — encoding corruption like `JoÃ«l` is repaired to `Joël` before any comparison, so the same person across two files always gets the same USER_N
- **Skip-already-processed files** — a `processed_files.json` registry tracks every file by its content hash; re-running only processes new files
- **Append mode** — new transactions are appended to the existing `master_transactions.csv` instead of overwriting it

### How to use
1. Drop new SMS export files into `data/`
2. Run all cells — the notebook will pause at each new file and ask you to confirm the detected name and phone
3. Already-processed files are silently skipped
4. Run the cleaning notebook afterward to regenerate `cleaned_data.csv`

> **Tip:** You only need to confirm each file once. After that, re-running is fully automatic for those files.

## ⚙️ Step 1 — Installs & Imports

We import everything the notebook needs. Note `hashlib` — this is the standard Python library for computing file hashes (fingerprints). No extra install needed.

In [16]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'openpyxl', 'chardet', '--quiet'], check=False)

import re, json, hashlib, unicodedata, datetime
from collections import Counter
from pathlib import Path
from IPython.display import display, clear_output

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('✅ All imports ready!')


✅ All imports ready!


## 📂 Step 2 — Folder Setup

Three important files live in `output/`:
- `master_transactions.csv` — all processed transactions stacked together
- `owner_map.json` — maps USER_N aliases to real names/phones (keep private)
- `processed_files.json` — the registry of already-processed files; **do not delete this** or the notebook will re-process everything
- `demographics_template.csv` — pre-filled with USER IDs for you to complete

In [17]:
NOTEBOOK_DIR  = Path().resolve()
DATA_DIR      = NOTEBOOK_DIR / 'data'
OUTPUT_DIR    = NOTEBOOK_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAP_FILE       = OUTPUT_DIR / 'owner_map.json'
MASTER_FILE    = OUTPUT_DIR / 'master_transactions.csv'
REGISTRY_FILE  = OUTPUT_DIR / 'processed_files.json'
DEMO_TEMPLATE  = OUTPUT_DIR / 'demographics_template.csv'

SUPPORTED = {'.csv', '.xlsx', '.xls'}
all_files = sorted(f for f in DATA_DIR.iterdir()
                   if f.is_file() and f.suffix.lower() in SUPPORTED)

if not all_files:
    print(f'⚠️  No files found in {DATA_DIR}')
else:
    print(f'📁 DATA_DIR   : {DATA_DIR}')
    print(f'📁 OUTPUT_DIR : {OUTPUT_DIR}')
    print(f'📄 Files found: {len(all_files)}')
    for f in all_files:
        print(f'   • {f.name}')


📁 DATA_DIR   : C:\Users\joelf\Documents\GitHub\mobile_money_analysis\data
📁 OUTPUT_DIR : C:\Users\joelf\Documents\GitHub\mobile_money_analysis\output
📄 Files found: 24
   • Messages avec MobileMoney 2026-03-20 21-32-36 - Madeleine Tchayo.csv
   • Messages avec MobileMoney 2026-03-22 12_27_25 - Jacques Fah.csv
   • Messages avec OrangeMoney 2026-03-20 21-31-49 - Madeleine Tchayo.csv
   • Messages avec OrangeMoney 2026-03-20 22_05_33 - Pascal Esaïe BOUGONG A ABEGA.csv
   • Messages avec OrangeMoney 2026-03-21 11_16_17 - DAVIS JERRY.csv
   • Messages avec OrangeMoney 2026-03-22 12_28_16 - Jacques Fah.csv
   • Messages avec OrangeMoney 2026-03-23 014528 - Manuel Karim.csv
   • Messages avec OrangeMoney Brown 2026-03-23 083311 - Brown Takou.csv
   • Messages with MobileMoney 2026-03-17 214651.xlsx
   • Messages with MobileMoney 2026-03-20 21_13_48 - Dejon Fah Joël Xavier.csv
   • Messages with MobileMoney 2026-03-23 073413.csv
   • Messages with MobileMoney 2026-03-23 210438.xlsx
   • Messa

## 📋 Step 3 — File Registry (Skip Already-Processed Files)

**What is a hash?** A hash is a fixed-length 'fingerprint' computed from a file's contents. The same file always produces the same hash. If even one character changes, the hash is completely different. This means we can reliably tell whether a file has been processed before without storing the whole file — just its fingerprint.

We use SHA-256 (a standard cryptographic hash function) and take only the first 16 characters — still effectively impossible to collide by accident.

In [18]:
def file_hash(path: Path) -> str:
    """Compute a 16-char SHA-256 fingerprint of a file's contents."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()[:16]


def load_registry() -> dict:
    """Load the processed-files registry, or start a fresh one."""
    if REGISTRY_FILE.exists():
        with open(REGISTRY_FILE, encoding='utf-8') as f:
            return json.load(f)
    return {'processed_files': []}


def save_registry(registry: dict):
    with open(REGISTRY_FILE, 'w', encoding='utf-8') as f:
        json.dump(registry, f, ensure_ascii=False, indent=2)


def is_already_processed(path: Path, registry: dict) -> bool:
    """Return True if this file's hash is in the registry."""
    h = file_hash(path)
    hashes = {r['hash'] for r in registry['processed_files']}
    return h in hashes


# Load the registry and show status
registry = load_registry()
already_done = sum(1 for f in all_files if is_already_processed(f, registry))
new_files    = [f for f in all_files if not is_already_processed(f, registry)]

print(f'Registry: {len(registry["processed_files"])} file(s) previously processed')
print(f'Already done  : {already_done} file(s) — will be skipped')
print(f'New to process: {len(new_files)} file(s)')
print()
if new_files:
    print('New files to process:')
    for f in new_files:
        print(f'  • {f.name}')
else:
    print('Nothing new to process. Add files to data/ and re-run.')


Registry: 15 file(s) previously processed
Already done  : 15 file(s) — will be skipped
New to process: 9 file(s)

New files to process:
  • Messages with MobileMoney 2026-03-17 214651.xlsx
  • Messages with MobileMoney 2026-03-23 073413.csv
  • Messages with MobileMoney 2026-03-23 210438.xlsx
  • Messages with MobileMoney 2026-03-24 134611.xlsx
  • Messages with MobileMoney 2026-03-24 193120 (messages from 3-24-2025 to 3-24-2026).xlsx
  • Messages with MobileMoney 2026-03-26 202316.csv
  • Messages with MobileMoney 2026-03-27 161511.xlsx
  • Messages with MTN MoMo 2026-03-21 13_56_46 - Annick Cindy Fah.csv
  • MOMO Mecos - Michelle Mfanga.xlsx


## 📖 Step 4 — File Loading Utilities

These functions handle the two hardest parts of reading SMS exports:

**Encoding detection:** French messages use accented characters (é, ë, ç…). If Python reads a file with the wrong encoding, these characters get corrupted into garbage like `JoÃ«l`. We use `chardet` to guess the encoding from the raw bytes, then verify by actually reading the file.

**Robust CSV parsing:** SMS messages contain commas (e.g. 'Fees: 50 FCFA, ID: 123, Ref: ABC'). A standard CSV parser would split these into extra columns and crash. Our parser splits from the left up to N-2 commas, then peels the last column (Type) from the right — keeping Content intact.

In [19]:
COL_ALIASES = {
    'date':      ['date'],
    'time':      ['heure', 'time'],
    'direction': ['direction'],
    'contact':   ['contact'],
    'phone':     ['telephone', 'phone'],
    'content':   ['contenu', 'content'],
    'type':      ['type'],
}

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    for canonical, aliases in COL_ALIASES.items():
        for col in df.columns:
            col_norm = (unicodedata.normalize('NFKD', str(col).strip().lower())
                        .encode('ascii', 'ignore').decode())
            if col_norm in aliases:
                rename[col] = canonical
                break
    return df.rename(columns=rename)

ENCODINGS = ['utf-8-sig', 'utf-8', 'latin-1', 'cp1252']

def detect_encoding(path: Path) -> str:
    try:
        import chardet
        result = chardet.detect(path.read_bytes()[:8192])
        if result.get('encoding') and result['confidence'] > 0.7:
            return result['encoding']
    except ImportError:
        pass
    for enc in ENCODINGS:
        try:
            with open(path, encoding=enc) as f:
                f.read(4096)
            return enc
        except (UnicodeDecodeError, LookupError):
            continue
    return 'latin-1'

def read_csv_robust(path: Path, encoding: str, skiprows: int) -> pd.DataFrame:
    with open(path, encoding=encoding, errors='replace') as f:
        all_lines = f.readlines()
    data_lines = all_lines[skiprows:]
    if not data_lines:
        raise pd.errors.EmptyDataError('No data after header skip')
    header_raw = data_lines[0].rstrip('\r\n')
    sep    = '\t' if '\t' in header_raw else ','
    header = [h.strip().strip('"') for h in header_raw.split(sep)]
    ncols  = len(header)
    rows   = []
    for line in data_lines[1:]:
        line = line.rstrip('\r\n')
        if not line.strip(): continue
        if sep == ',':
            parts = line.split(',', ncols - 2)
            if len(parts) == ncols - 1:
                lc = parts[-1].rfind(',')
                if lc != -1:
                    parts = parts[:-1] + [parts[-1][:lc], parts[-1][lc+1:]]
        else:
            parts = line.split('\t')
        parts = (parts + [''] * ncols)[:ncols]
        rows.append([p.strip().strip('"') for p in parts])
    return pd.DataFrame(rows, columns=header)

EXPECTED_COLS = {'content', 'contenu', 'date'}
CSV_SKIP = 3

def load_file(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext == '.csv':
        encoding = detect_encoding(path)
        for skip in (CSV_SKIP, 0):
            try:
                df = read_csv_robust(path, encoding, skip)
            except pd.errors.EmptyDataError:
                continue
            df = normalize_columns(df)
            if EXPECTED_COLS.intersection(df.columns): break
        else:
            raise ValueError(f'Could not find expected columns in {path.name!r}')
    elif ext in ('.xlsx', '.xls'):
        for skip in (CSV_SKIP, 0):
            df = pd.read_excel(path, skiprows=skip, header=0, dtype=str)
            df = normalize_columns(df)
            if EXPECTED_COLS.intersection(df.columns): break
        else:
            raise ValueError(f'Could not find expected columns in {path.name!r}')
    else:
        raise ValueError(f'Unsupported file type: {ext}')
    df['content'] = (
        df['content'].astype(str)
        .str.replace('_x000d_', ' ', regex=False).str.strip()
    )
    return df

print('✅ File loading utilities ready!')


✅ File loading utilities ready!


## 🔤 Step 5 — Name Normalisation & Mojibake Fix

**What is mojibake?** It's the garbled text you get when a file encoded in UTF-8 is read as if it were Latin-1. The French letter `ë` (U+00EB) is stored as two bytes `0xC3 0xAB` in UTF-8. Read as Latin-1, those two bytes become the characters `Ã` and `«` — producing `JoÃ«l`.

**The fix:** We try to re-encode the broken string back to Latin-1 bytes, then decode those bytes as UTF-8. If that produces clean text, we use it. If it produces garbage or fails, we keep the original.

**Why this matters:** Without this fix, `Joël` and `JoÃ«l` normalise to different keys and the same person gets two USER_N IDs. With the fix, both normalise to `joel` and correctly map to one person.

In [20]:
def fix_mojibake(s: str) -> str:
    """
    Repair encoding corruption: 'JoÃ«l' -> 'Joël'.
    Only applies the fix if the result is printable and plausible.
    """
    if not isinstance(s, str): return s
    try:
        fixed = s.encode('latin-1').decode('utf-8')
        # Accept only if no control characters crept in
        if all(unicodedata.category(c) != 'Cc' for c in fixed):
            return fixed
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass
    return s


def norm_name(s: str) -> str:
    """
    Normalise a name for consistent comparison:
    1. Fix mojibake ('JoÃ«l' -> 'Joël')
    2. Strip accents ('Joël' -> 'Joel')
    3. Lowercase and collapse whitespace
    Result: 'Dejon Fah JoÃ«l Xavier' == 'Dejon Fah Joël Xavier' == 'dejon fah joel xavier'
    """
    s = fix_mojibake(str(s))
    nfkd = unicodedata.normalize('NFKD', s)
    ascii_str = nfkd.encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', ascii_str.strip().lower())


# Quick self-test
_tests = [
    ('Dejon Fah JoÃ«l Xavier', 'Dejon Fah Joël Xavier'),
    ('TCHAYO',                 'TCHAYO'),
]
for corrupted, clean in _tests:
    assert norm_name(corrupted) == norm_name(clean), \
        f'Mismatch: {norm_name(corrupted)!r} != {norm_name(clean)!r}'

print('✅ Name normalisation with mojibake fix ready!')
print("   'Dejon Fah JoÃ«l Xavier' normalises to:",
      norm_name('Dejon Fah JoÃ«l Xavier'))


✅ Name normalisation with mojibake fix ready!
   'Dejon Fah JoÃ«l Xavier' normalises to: dejon fah joel xavier


## 🔍 Step 6 — Operator & Owner Detection

**Owner phone:** The account holder's phone number appears in every balance confirmation, every transfer notification, every recharge receipt. It is by far the most frequently mentioned number in their own SMS export. We count every phone number across all messages and take the top one.

**Owner name:** Once we know the phone, we scan for messages where that phone appears next to a name (MTN format: `'NAME (237XXXXXXXXX)'`, OM format: `'XXXXXXXXX NAME to …'`). The first match we find is the owner's name.

**Filename fallback:** If neither pattern fires (e.g. very small dataset, unusual format), we extract the name from the filename itself. SMS Exporter always names files like `Messages_with_OrangeMoney_DATE_-_First_Last.csv`.

In [21]:
def detect_operator(df: pd.DataFrame) -> str:
    for col in ['contact', 'phone']:
        if col in df.columns:
            s = df[col].dropna().astype(str).str.lower()
            if s.str.contains('orangemoney', na=False).any():
                return 'OrangeMoney'
            if s.str.contains('mobilemoney', na=False).any():
                return 'MobileMoney'
    return 'Unknown'

PHONE_RE_DETECT = re.compile(r'\b(237)?(6\d{8}|2\d{8})\b')
MTN_OWNER_RE = re.compile(
    r'(\b[A-ZÀ-Ýa-zà-ý][A-ZÀ-Ýa-zà-ý]+'
    r'(?:\s+[A-ZÀ-Ýa-zà-ý&][A-ZÀ-Ýa-zà-ý&]+)*)'
    r'\s+\(237(\d{9})\s*\)'
)
OM_OWNER_RE = re.compile(
    r'\b((?:237)?(?:6\d{8}|2\d{8}))\s+'
    r'((?:[A-ZÀ-Ýa-zà-ý&][A-ZÀ-Ýa-zà-ý&]+)'
    r'(?:\s+(?!to\b|vers\b|avec\b|reussi\b|Informations\b)'
    r'[A-ZÀ-Ýa-zà-ý&][A-ZÀ-Ýa-zà-ý&]+){0,4})'
    r'(?=\s+(?:to|vers|avec|reussi|Informations)|\s*[.\n,])'
)
NAME_STOPWORDS = {'from','to','of','by','de','du','par','vers','avec',
                  'le','la','les','un','une','your','xaf','fcfa'}

def bare_phone(raw: str) -> str:
    s = str(raw).strip().replace(' ', '')
    return s[3:] if s.startswith('237') and len(s) == 12 else s

def clean_name_token(raw: str) -> str:
    words = raw.split()
    while words and words[0].lower() in NAME_STOPWORDS:
        words = words[1:]
    return ' '.join(words)

def detect_owner_phone(df: pd.DataFrame) -> str | None:
    c: Counter = Counter()
    for text in df['content'].dropna():
        for m in PHONE_RE_DETECT.finditer(str(text)):
            c[m.group(2)] += 1
    return c.most_common(1)[0][0] if c else None

def detect_owner_name(df: pd.DataFrame,
                      owner_phone: str | None) -> tuple[str | None, str]:
    """Returns (name, source_description)."""
    if not owner_phone: return None, 'not found'
    for text in df['content'].dropna():
        text = fix_mojibake(str(text))  # fix encoding before regex scan
        for m in MTN_OWNER_RE.finditer(text):
            if bare_phone('237' + m.group(2)) == owner_phone:
                n = clean_name_token(m.group(1))
                if n: return n, 'MTN message pattern'
        for m in OM_OWNER_RE.finditer(text):
            if bare_phone(m.group(1)) == owner_phone:
                n = clean_name_token(m.group(2))
                if n: return n, 'OM message pattern'
    return None, 'not found in messages'

def detect_owner_from_filename(path: Path) -> str | None:
    stem = path.stem
    for sep in (' - ', '_-_'):
        if sep in stem:
            raw = stem.rsplit(sep, 1)[1].replace('_', ' ').strip()
            return fix_mojibake(raw)
    return None

print('✅ Owner detection ready!')


✅ Owner detection ready!


## ✋ Step 7 — Interactive Owner Confirmation

This is the most important new feature in v3. For every new file, the notebook **pauses** and shows you what it auto-detected. You then choose one of five actions:

| Choice | Action |
|--------|--------|
| `1` | Confirm — everything is correct, continue |
| `2` | Edit name — phone is right, name needs correction |
| `3` | Edit phone — name is right, phone needs correction |
| `4` | Edit both — neither was detected correctly |
| `5` | Skip — don't process this file at all |

**When is option 4 forced?** If both name AND phone are missing, the notebook shows a warning and will not let you press 1 until you've provided at least a name. This prevents ghost users with no identity.

**Important:** Enter phone numbers as 9 digits without the country code (e.g. `656997810` not `237656997810`).

In [22]:
def confirm_owner(filepath_name: str, detected_name: str | None,
                  detected_phone: str | None, detected_operator: str,
                  detection_source: str) -> tuple[str, str, bool]:
    """
    Pause and ask the user to confirm auto-detected owner information.

    Returns:
        confirmed_name  (str)  — the name to use for this file's owner
        confirmed_phone (str)  — the phone to use
        should_skip     (bool) — True if user chose to skip this file
    """
    name  = detected_name  or ''
    phone = detected_phone or ''

    print(f'\n{"─"*62}')
    print(f'  📄 File: {filepath_name}')
    print(f'{"─"*62}')
    print(f'  Auto-detected:')
    print(f'    Operator : {detected_operator}')
    print(f'    Name     : {name or "(not found)"}')
    print(f'    Phone    : {phone or "(not found)"}')
    print(f'    Source   : {detection_source}')
    print()

    if not name and not phone:
        print('  ⚠️  Auto-detection failed completely.')
        print('     You must enter a name (option 4) before continuing.')

    print('  [1] ✅ Confirm and continue')
    print('  [2] ✏️  Edit name only')
    print('  [3] ✏️  Edit phone number only')
    print('  [4] ✏️  Edit both name and phone')
    print('  [5] ⏭️  Skip this file')
    print()

    while True:
        choice = input('  Your choice [1-5]: ').strip()

        if choice == '1':
            if not name:
                print('  ⚠️  Name is empty — please use option 2 or 4 to provide one.')
                continue
            print(f'  ✅ Confirmed: {name} / {phone or "(no phone)"}')
            return name, phone, False

        elif choice == '2':
            name = input('  Enter correct full name: ').strip()
            if not name:
                print('  ⚠️  Name cannot be empty. Try again.')
                continue
            print(f'  ✅ Updated name: {name}')
            return name, phone, False

        elif choice == '3':
            phone = input('  Enter phone (9 digits, no country code): ').strip()
            print(f'  ✅ Updated phone: {phone}')
            return name, phone, False

        elif choice == '4':
            name = input('  Enter correct full name: ').strip()
            if not name:
                print('  ⚠️  Name cannot be empty. Try again.')
                continue
            phone = input('  Enter phone (9 digits, no country code): ').strip()
            print(f'  ✅ Updated: {name} / {phone}')
            return name, phone, False

        elif choice == '5':
            print('  ⏭️  File skipped.')
            return name, phone, True

        else:
            print('  ⚠️  Please enter a number between 1 and 5.')

print('✅ Interactive confirmation ready!')


✅ Interactive confirmation ready!


## 🧠 Step 8 — Filter, Extract & Classify Transactions

These three functions work together in the pipeline:

- **`filter_balance_messages`** — keeps only messages that contain a balance figure (`nouveau solde` / `new balance`), discarding OTP codes, promotions, and other noise
- **`extract_amount` / `extract_new_balance`** — pulls the numbers out of the message text using regex patterns that cover both French and English phrasing
- **`classify_transaction`** — matches each message against 8 transaction types (depot, retrait, transfert IN/OUT, paiement, rechargement, airtime, transaction) in priority order

In [23]:
# ── Balance filter ───────────────────────────────────────────────────────
BALANCE_RE = re.compile(
    r'nouveau\s+solde|nouveau\s+solde\s+est|new\s+balance|your\s+new\s+balance',
    re.IGNORECASE
)

def filter_balance_messages(df: pd.DataFrame) -> pd.DataFrame:
    mask = df['content'].apply(lambda x: bool(BALANCE_RE.search(str(x))))
    return df[mask].copy().reset_index(drop=True)


# ── Amount / balance extraction ───────────────────────────────────────────
AMOUNT_RE          = re.compile(
    r'(?:montant[^:]*:\s*|(?<!solde\sest\s)de\s+|amount\s+|of\s+)'
    r'(\d[\d\s,\.]*?)\s*(FCFA|XAF)', re.IGNORECASE)
AMOUNT_FALLBACK_RE = re.compile(r'(\d[\d\s]*?)\s*(FCFA|XAF)', re.IGNORECASE)
BALANCE_VAL_RE     = re.compile(
    r'(?:votre\s+nouveau\s+solde\s+est\s+de\s+|your\s+new\s+balance\s+is\s+|'
    r'nouveau\s+solde\s+est\s+de\s*:?|new\s+balance\s+is\s*:?|'
    r'nouveau\s+solde\s+est\s*:?|new\s+balance\s+is\s*:?|'
    r'nouveau\s+solde\s*:?|new\s+balance\s*:?|solde\s*:?|balance\s*:?)'
    r'\s*(\d[\d\s,\.]*?)\s*(FCFA|XAF)', re.IGNORECASE)

def _clean_num(raw) -> float | None:
    if raw is None: return None
    try:    return float(re.sub(r'[\s,]', '', str(raw)))
    except: return None

def extract_amount(text: str):
    m = AMOUNT_RE.search(text) or AMOUNT_FALLBACK_RE.search(text)
    return (_clean_num(m.group(1)), m.group(2).upper()) if m else (None, None)

def extract_new_balance(text: str):
    m = BALANCE_VAL_RE.search(text)
    return (_clean_num(m.group(1)), m.group(2).upper()) if m else (None, None)


# ── Transaction classification ────────────────────────────────────────────
TX_RULES = [
    ('retrait', 'OUT', [
        r"retrait\s+d'argent", r'retrait\s+de\s+\d',
        r'vous\s+avez\s+effectue\s+avec\s+succes\s+le\s+retrait',
        r'withdrawal\s+successful', r'cash\s+out',
        r'you\s+have\s+successfully\s+withdrawn',
        r'withdrawn\s+from\s+your\s+mobile\s+money',
        r'you\s+have\s+withdrawn\s+\d+\s*(xaf|fcfa)',
        r'have\s+via\s+agent.*withdrawn',
    ]),
    ('depot', 'IN', [
        r'depot\s+effectue\s+par', r'deposit\s+made\s+by',
        r'deposit\s+to\s+your', r'you\s+have\s+received\s+a\s+deposit\s+of',
    ]),
    ('transfert', 'IN', [
        r'vous\s+avez\s+re[cç]u\s+\d',
        r'vous\s+avez\s+recu\s+avec\s+succes',
        r'you\s+have\s+received\s+\d',
        r'has\s+been\s+added\s+to\s+your',
        r'adjustment\s+has\s+been\s+made',
        r'you\s+just\s+received',
        r'you\s+have\s+received.*in\s+your\s+mobile\s+money',
        r'vous\s+avez\s+re[cç]u.*xaf',
    ]),
    ('transfert', 'OUT', [
        r'transfert\s+de\s+\d+\s*(fcfa|xaf)',
        r'transfer\s+of\s+\d',
        r'transfert.*effectue.*succes.*\d',
        r'transfert.*vers\s+\d',
        r'successful\s+transfer\s+.*\s+xaf\s+to',
        r'you\s+have\s+transferred\s+\d',
    ]),
    ('paiement', 'OUT', [
        r'paiement\s+de\s+votre\s+facture',
        r'votre\s+paiement\s+de\s+\d',
        r'your\s+payment\s+of\s+\d',
        r'paiement.*r[ée]ussi', r'payment.*successful',
        r'paiement\s+total',
        r'vous\s+venez\s+d.effectuer\s+un\s+pa[yi]e?ment',
        r'has\s+been\s+completed',
    ]),
    ('rechargement', 'OUT', [
        r'rechargement\s+reussi', r'top.?up\s+successful',
        r'recharge\s+successful',
    ]),
    ('airtime', 'OUT', [
        r'achete\s+avec\s+succes.*airtime', r'airtime.*transaction',
        r'paiement.*airtime', r'payment.*airtime',
        r'you\s+have\s+received.*airtime\s+from',
        r'recu.*xaf\s+airtime', r'received.*xaf\s+airtime',
    ]),
    ('transaction', 'OUT', [
        r'une\s+transaction\s+de\s+\d',
        r'a\s+transaction\s+of\s+\d',
        r'transaction.*effectuee\s+par',
        r'transaction.*made\s+by',
        r'from\s+your\s+mobile\s+money\s+account\s+by',
        r'from\s+your\s+mobile\s+money\s+account.*completed',
    ]),
]
TX_RULES_COMPILED = [
    (t, d, [re.compile(p, re.IGNORECASE) for p in pats])
    for t, d, pats in TX_RULES
]

def classify_transaction(text: str) -> tuple[str, str]:
    for tx_type, direction, compiled in TX_RULES_COMPILED:
        for pat in compiled:
            if pat.search(text):
                return tx_type, direction
    return 'autre', 'unknown'

print('✅ Filter, extraction & classification ready!')


✅ Filter, extraction & classification ready!


## 🔒 Step 9 — Anonymization Engine

The owner is replaced with `[USER_N]` and `[USER_N_phone]`. Third parties get hashed tokens: `[CONTACT_XXXX]` for names, `[PHONE_XXXX]` for numbers. The hash ensures the same contact always gets the same token within a file.

In [24]:
import hashlib as _hl

def short_hash(value: str, length: int = 4) -> str:
    return _hl.md5(str(value).encode()).hexdigest()[:length].upper()

PHONE_RE         = re.compile(r'\b\d{9,12}\b')
NAME_AFTER_RE    = re.compile(
    r'\b(\d{9,12})\s*[-–:]?\s*'
    r'([A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}(?:\s+[A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}){0,3})\b')
NAME_BEFORE_RE   = re.compile(
    r'\b([A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}(?:\s+[A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}){0,3})'
    r'\s*\((\d{9,12}\s*)\)')
WITHDRAWAL_RE    = re.compile(
    r'\b(?:withdrawn|withdraw|retrait)\b.*?\b(?:chez|at)\s*[:\-–]?\s*'
    r'([A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}(?:\s+[A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}){0,4})\b',
    re.IGNORECASE)
SYSTEM_TOKENS    = {
    'FCFA','XAF','SMS','ID','MTN','ORANGE','MOBILEMONEY','OM','MOMO',
    'OTP','PIN','ENEO','CAMWATER','CANAL','DSTV','YELLO','MTNC',
}

def _phone_variants(phone: str) -> list[str]:
    if not phone: return []
    v = {phone}
    if re.match(r'^6\d{8}$', phone): v.add('237' + phone)
    if re.match(r'^237\d{9}$', phone): v.add(phone[3:])
    return list(v)

def anonymize_message(text: str, user_name: str | None,
                      user_phone: str | None, user_id: str) -> str:
    r = fix_mojibake(str(text))  # fix encoding in the message itself too
    if user_name and user_name.strip():
        r = re.sub(re.escape(user_name.strip()), f'[{user_id}]',
                   r, flags=re.IGNORECASE)
    def _wd(m):
        nm = m.group(1).strip()
        return m.group(0).replace(nm, f'[CONTACT_{short_hash(nm)}]')
    r = WITHDRAWAL_RE.sub(_wd, r)
    r = NAME_AFTER_RE.sub(
        lambda m: f'{m.group(1)} [CONTACT_{short_hash(m.group(2).strip())}]', r)
    r = NAME_BEFORE_RE.sub(
        lambda m: f'[CONTACT_{short_hash(m.group(1).strip())}] ({m.group(2).strip()})', r)
    for v in _phone_variants(user_phone):
        r = r.replace(v, f'[{user_id}_phone]')
    r = PHONE_RE.sub(lambda m: f'[PHONE_{m.group(0)[-4:]}]', r)
    def _caps(m):
        nm = m.group(0).strip()
        ws = nm.split()
        if len(ws) < 2 or any(w in SYSTEM_TOKENS for w in ws): return nm
        return f'[CONTACT_{short_hash(nm)}]'
    r = re.sub(
        r'\b([A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}(?:\s+[A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}){1,3})\b',
        _caps, r)
    return r

print('✅ Anonymization engine ready!')


✅ Anonymization engine ready!


## 💾 Step 10 — Styled Excel Export

In [25]:
def thin_border(color='BBBBBB'):
    s = Side(style='thin', color=color)
    return Border(left=s, right=s, top=s, bottom=s)

COL_WIDTHS = {
    'UserId':14,'Date':12,'Time':10,'Operator':14,
    'Transaction_type':16,'Direction':10,'Amount':11,
    'Currency':9,'New_balance':13,'Anonymized_Content':70,
}

def export_xlsx(df: pd.DataFrame, path: Path,
                user_id: str, operator: str):
    wb = Workbook(); ws = wb.active
    ws.title = user_id[:31]; ws.sheet_view.showGridLines = False
    n = len(df.columns)
    ws.merge_cells(f'A1:{get_column_letter(n)}1')
    ws['A1'] = f'Anonymized Mobile Money - {user_id} ({operator})'
    ws['A1'].font = Font(name='Arial',bold=True,size=13,color='FFFFFF')
    ws['A1'].fill = PatternFill('solid',start_color='1F3864')
    ws['A1'].alignment = Alignment(horizontal='center',vertical='center')
    ws.row_dimensions[1].height = 30
    for ci,col in enumerate(df.columns,1):
        c = ws.cell(row=2,column=ci)
        c.value=col; c.font=Font(name='Arial',bold=True,size=9,color='FFFFFF')
        c.fill=PatternFill('solid',start_color='2E75B6')
        c.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
        c.border=thin_border()
    ws.row_dimensions[2].height = 28
    for ri,row_data in enumerate(df.itertuples(index=False),3):
        d = str(row_data.Direction).upper()
        for ci,val in enumerate(row_data,1):
            col_name=df.columns[ci-1]; is_c=(col_name=='Anonymized_Content')
            c=ws.cell(row=ri,column=ci)
            c.value=val if val is not None else ''
            c.font=Font(name='Arial',size=9); c.border=thin_border()
            c.alignment=Alignment(horizontal='left' if is_c else 'center',
                                  vertical='center',wrap_text=is_c)
            if col_name=='Direction':
                bg='C6EFCE' if d=='IN' else 'FFC7CE'
                c.fill=PatternFill('solid',start_color=bg)
                c.font=Font(name='Arial',size=9,bold=True)
            elif col_name=='Transaction_type':
                c.fill=PatternFill('solid',start_color='EBF3FB')
            else:
                c.fill=PatternFill('solid',
                    start_color='F2F9FF' if ri%2==0 else 'FFFFFF')
        ws.row_dimensions[ri].height=15
    for ci,col in enumerate(df.columns,1):
        ws.column_dimensions[get_column_letter(ci)].width=COL_WIDTHS.get(col,14)
    ws.freeze_panes='A3'; wb.save(path)

print('✅ Excel export ready!')


✅ Excel export ready!


## 🚀 Step 11 — Main Pipeline

This is the orchestration cell. It ties everything together:

1. **Load the registry** — find out which files are already processed
2. **For each new file:** load → detect operator + owner → ask you to confirm
3. **If confirmed:** classify transactions → anonymize → save individual files
4. **Append** new rows to `master_transactions.csv` (not overwrite)
5. **Update the registry** — mark each processed file so it's skipped next time
6. **Update `owner_map.json`** — add new user entries

**Note about the owner registry:** Two files from the same person (e.g. one OM + one MTN file) get the same USER_N because `norm_name()` produces the same key for both, and `owner_registry` is checked before creating a new ID.

In [26]:
# ── Load persistent state from previous runs ─────────────────────────────
registry = load_registry()

# Owner registry: norm_name -> (USER_N, USER_N_phone)
# Loaded from owner_map.json so IDs are consistent across runs
owner_registry: dict[str, tuple[str, str]] = {}
if MAP_FILE.exists():
    with open(MAP_FILE, encoding='utf-8') as f:
        existing_map = json.load(f)
    for r in existing_map.get('owner_map', []):
        if r.get('original_name'):
            key = norm_name(r['original_name'])
            owner_registry[key] = (r['user_alias'], r['phone_alias'])

# Set counter to continue from last USER_N
_user_ctr = max(
    (int(v[0].split('_')[1]) for v in owner_registry.values()),
    default=0
)

def get_owner_alias(name_norm: str) -> tuple[str, str]:
    global _user_ctr
    if name_norm not in owner_registry:
        _user_ctr += 1
        owner_registry[name_norm] = (
            f'USER_{_user_ctr}', f'USER_{_user_ctr}_phone'
        )
    return owner_registry[name_norm]


# ── Filter to only new files ──────────────────────────────────────────────
new_files = [f for f in all_files
             if not is_already_processed(f, registry)]
skipped_files = [f for f in all_files
                 if is_already_processed(f, registry)]

print(f'Files already processed (skipping): {len(skipped_files)}')
for f in skipped_files:
    print(f'  ⏭️  {f.name}')
print(f'\nNew files to process: {len(new_files)}')

if not new_files:
    print('\n✅ Nothing to do. Add new files to data/ and re-run.')
else:
    print('─' * 62)
    print('  For each file below, review the auto-detected information')
    print('  and confirm or correct it before processing begins.')
    print('─' * 62)


# ── Process each new file ─────────────────────────────────────────────────
new_owner_records = []
all_new_dfs       = []

for filepath in new_files:

    # ── 1. Load ───────────────────────────────────────────────────────────
    try:
        df_raw = load_file(filepath)
    except Exception as e:
        print(f'\n❌ Could not load {filepath.name}: {e}')
        print('   Skipping this file.')
        continue

    # ── 2. Auto-detect operator + owner ───────────────────────────────────
    operator    = detect_operator(df_raw)
    owner_phone = detect_owner_phone(df_raw)
    owner_name, source = detect_owner_name(df_raw, owner_phone)

    # Filename fallback if message detection failed
    if not owner_name:
        fn_name = detect_owner_from_filename(filepath)
        if fn_name:
            owner_name = fn_name
            source     = 'filename'

    # ── 3. Interactive confirmation ───────────────────────────────────────
    confirmed_name, confirmed_phone, should_skip = confirm_owner(
        filepath.name, owner_name, owner_phone, operator, source
    )
    if should_skip:
        print(f'   File {filepath.name} skipped by user.')
        continue

    # ── 4. Assign USER_N (reuse if same person appeared before) ──────────
    owner_norm          = norm_name(confirmed_name)
    user_id, phone_alias = get_owner_alias(owner_norm)
    print(f'   Assigned: {user_id}')

    # ── 5. Filter, extract, classify, anonymize ───────────────────────────
    df_filtered = filter_balance_messages(df_raw)
    n_total     = len(df_raw)
    n_filtered  = len(df_filtered)
    print(f'   Messages: {n_total} total → {n_filtered} balance-related')

    if n_filtered == 0:
        print('   ⚠️  No balance-related messages. Skipping.')
        continue

    records = []
    for _, row in df_filtered.iterrows():
        text = str(row['content'])
        tx_type, direction    = classify_transaction(text)
        amount, currency      = extract_amount(text)
        new_balance, bal_curr = extract_new_balance(text)
        if currency is None: currency = bal_curr
        anon = anonymize_message(text, confirmed_name,
                                 confirmed_phone, user_id)
        records.append({
            'UserId':             user_id,
            'Date':               row.get('date', ''),
            'Time':               row.get('time', ''),
            'Operator':           operator,
            'Transaction_type':   tx_type,
            'Direction':          direction,
            'Amount':             amount,
            'Currency':           currency,
            'New_balance':        new_balance,
            'Anonymized_Content': anon,
        })

    df_out  = pd.DataFrame(records)
    n_in    = (df_out['Direction'] == 'IN').sum()
    n_out   = (df_out['Direction'] == 'OUT').sum()
    n_autre = (df_out['Transaction_type'] == 'autre').sum()
    print(f'   Extracted: {len(df_out)} tx | '
          f'{n_in} IN | {n_out} OUT | {n_autre} unclassified')

    # ── 6. Save individual files ──────────────────────────────────────────
    stem      = filepath.stem
    sep_token = ' - ' if ' - ' in stem else ('_-_' if '_-_' in stem else None)
    new_stem  = (stem.rsplit(sep_token,1)[0] + sep_token + user_id
                 if sep_token else f'{user_id}_{operator}')
    # Dedup guard: same owner + same operator -> append _2, _3...
    base_xlsx = OUTPUT_DIR / (new_stem + '.xlsx')
    base_csv  = OUTPUT_DIR / (new_stem + '.csv')
    ctr = 1
    while base_xlsx.exists() or base_csv.exists():
        base_xlsx = OUTPUT_DIR / (new_stem + f'_{ctr}.xlsx')
        base_csv  = OUTPUT_DIR / (new_stem + f'_{ctr}.csv')
        ctr += 1

    try:
        export_xlsx(df_out, base_xlsx, user_id, operator)
        df_out.to_csv(base_csv, index=False, encoding='utf-8-sig')
        print(f'   ✅ Saved → {base_xlsx.name}')
    except Exception as e:
        print(f'   ❌ Save error: {e}'); continue

    all_new_dfs.append(df_out)

    # ── 7. Mark file as processed in registry ─────────────────────────────
    registry['processed_files'].append({
        'filename':          filepath.name,
        'hash':              file_hash(filepath),
        'processed_at':      datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'user_alias':        user_id,
        'confirmed_name':    confirmed_name,
        'confirmed_phone':   confirmed_phone,
        'transactions_added': len(df_out),
    })
    save_registry(registry)

    # ── 8. Record for owner map ───────────────────────────────────────────
    new_owner_records.append({
        'user_alias':    user_id,
        'phone_alias':   phone_alias,
        'original_name': confirmed_name,
        'original_phone':confirmed_phone,
        'operator':      operator,
        'source_file':   filepath.name,
        'output_xlsx':   base_xlsx.name,
        'output_csv':    base_csv.name,
        'transactions':  len(df_out),
        'in_count':      int(n_in),
        'out_count':     int(n_out),
        'unclassified':  int(n_autre),
    })


print(f'\n{"─"*62}')
print(f'Processing complete. {len(all_new_dfs)} new file(s) processed.')


Files already processed (skipping): 15
  ⏭️  Messages avec MobileMoney 2026-03-20 21-32-36 - Madeleine Tchayo.csv
  ⏭️  Messages avec MobileMoney 2026-03-22 12_27_25 - Jacques Fah.csv
  ⏭️  Messages avec OrangeMoney 2026-03-20 21-31-49 - Madeleine Tchayo.csv
  ⏭️  Messages avec OrangeMoney 2026-03-20 22_05_33 - Pascal Esaïe BOUGONG A ABEGA.csv
  ⏭️  Messages avec OrangeMoney 2026-03-21 11_16_17 - DAVIS JERRY.csv
  ⏭️  Messages avec OrangeMoney 2026-03-22 12_28_16 - Jacques Fah.csv
  ⏭️  Messages avec OrangeMoney 2026-03-23 014528 - Manuel Karim.csv
  ⏭️  Messages avec OrangeMoney Brown 2026-03-23 083311 - Brown Takou.csv
  ⏭️  Messages with MobileMoney 2026-03-20 21_13_48 - Dejon Fah Joël Xavier.csv
  ⏭️  Messages with MobileMoney 2026-03-25 145028.xlsx
  ⏭️  Messages with MobileMoney 2026-03-26 162041.xlsx
  ⏭️  Messages with OrangeMoney 2026-03-20 212842 - Fah Rommel.csv
  ⏭️  Messages with OrangeMoney 2026-03-20 21_13_04 - Dejon Fah Joël Xavier.csv
  ⏭️  Messages with OrangeMoney 20

## 🔗 Step 12 — Append to Master File

**Why append and not overwrite?** If we overwrote `master_transactions.csv` each run, we would have to re-process every single file every time — defeating the purpose of the registry. Instead we read the existing master file (if it exists), stack the new rows on the bottom, and save. Old rows are untouched.

In [27]:
if all_new_dfs:
    df_new = pd.concat(all_new_dfs, ignore_index=True)
    df_new['Date']        = pd.to_datetime(df_new['Date'], format='mixed', dayfirst=True, errors='coerce')
    df_new['Amount']      = pd.to_numeric(df_new['Amount'], errors='coerce')
    df_new['New_balance'] = pd.to_numeric(df_new['New_balance'], errors='coerce')

    if MASTER_FILE.exists():
        df_existing = pd.read_csv(MASTER_FILE, parse_dates=['Date'])
        df_master   = pd.concat([df_existing, df_new], ignore_index=True)
        print(f'Appended {len(df_new)} rows to existing master '
              f'({len(df_existing)} rows)')
    else:
        df_master = df_new
        print(f'Created new master file with {len(df_master)} rows')

    df_master.to_csv(MASTER_FILE, index=False, encoding='utf-8-sig')
    print(f'✅ master_transactions.csv: '
          f'{len(df_master):,} rows, '
          f'{df_master["UserId"].nunique()} users')
else:
    print('No new data to append.')
    if MASTER_FILE.exists():
        df_master = pd.read_csv(MASTER_FILE)
        print(f'Existing master: {len(df_master):,} rows, '
              f'{df_master["UserId"].nunique()} users')


Appended 54 rows to existing master (7743 rows)
✅ master_transactions.csv: 7,797 rows, 16 users


## 🗂️ Step 13 — Update Owner Map & Demographics Template

`owner_map.json` is your **private** key that links USER_N aliases back to real people. It is updated incrementally — new entries are appended, existing ones kept.

The demographics template is also regenerated so it always contains all current USER IDs, including newly added ones.

In [28]:
# ── Update owner_map.json ────────────────────────────────────────────────
existing_map_records = []
if MAP_FILE.exists():
    with open(MAP_FILE, encoding='utf-8') as f:
        existing_map_records = json.load(f).get('owner_map', [])

# Merge: existing + new (avoid exact duplicate entries)
existing_files = {r['source_file'] for r in existing_map_records}
merged_records = existing_map_records + [
    r for r in new_owner_records if r['source_file'] not in existing_files
]

with open(MAP_FILE, 'w', encoding='utf-8') as f:
    json.dump({'owner_map': merged_records}, f, ensure_ascii=False, indent=2)
print(f'✅ owner_map.json: {len(merged_records)} total entries')


# ── Regenerate demographics template with all USER IDs ────────────────────
DEMO_COLS = [
    'UserId', 'Age_range', 'Gender', 'Occupation', 'Education_level',
    'Monthly_income_range', 'Geographic_zone', 'Household_size',
    'Primary_MM_use', 'Smartphone_ownership',
]

all_user_ids = sorted(
    {r['user_alias'] for r in merged_records},
    key=lambda x: int(x.split('_')[1])
)

df_template = pd.DataFrame({'UserId': all_user_ids})
for col in DEMO_COLS[1:]:
    df_template[col] = ''

# Merge with any existing demographics.csv
EXISTING_DEMO = NOTEBOOK_DIR / 'demographics.csv'
if EXISTING_DEMO.exists():
    df_ex = pd.read_csv(EXISTING_DEMO)
    df_ex.columns = df_ex.columns.str.strip()
    # Find the id column (handles 'userID', 'UserId', 'userid')
    id_col = next((c for c in df_ex.columns if c.lower() == 'userid'), None)
    if id_col:
        df_ex = df_ex.rename(columns={id_col: 'UserId'})
        df_template = df_template.merge(
            df_ex, on='UserId', how='left', suffixes=('', '_ex')
        )
        for col in DEMO_COLS[1:]:
            ex_col = f'{col}_ex'
            if ex_col in df_template.columns:
                df_template[col] = df_template[col].fillna(df_template[ex_col])
                df_template.drop(columns=[ex_col], inplace=True)
        filled = df_template['Gender'].replace('', float('nan')).notna().sum()
        print(f'Merged with demographics.csv: {filled}/{len(df_template)} users filled')

df_template.to_csv(DEMO_TEMPLATE, index=False, encoding='utf-8-sig')
print(f'✅ demographics_template.csv: {len(df_template)} users')
print()
display(df_template)


✅ owner_map.json: 16 total entries
✅ demographics_template.csv: 16 users



,UserId,Age_range,Gender,Occupation,Education_level,Monthly_income_range,Geographic_zone,Household_size,Primary_MM_use,Smartphone_ownership
0,USER_1,,,,,,,,,
1,USER_2,,,,,,,,,
2,USER_3,,,,,,,,,
3,USER_4,,,,,,,,,
4,USER_5,,,,,,,,,
5,USER_6,,,,,,,,,
6,USER_7,,,,,,,,,
7,USER_8,,,,,,,,,
8,USER_9,,,,,,,,,
9,USER_10,,,,,,,,,


In [ ]:
%%sql


## 📊 Step 14 — Run Summary

In [29]:
print('=' * 62)
print('  RUN SUMMARY')
print('=' * 62)
print(f'  Files in data/      : {len(all_files)}')
print(f'  Already processed   : {len(all_files) - len(new_files)}')
print(f'  Newly processed     : {len(all_new_dfs)}')
print(f'  Total in registry   : {len(registry["processed_files"])}')
if MASTER_FILE.exists():
    _m = pd.read_csv(MASTER_FILE)
    print(f'  Master rows         : {len(_m):,}')
    print(f'  Unique users        : {_m["UserId"].nunique()}')
print('=' * 62)
print()
print('Next steps:')
print('  1. Fill in demographics_template.csv and save as demographics.csv')
print('  2. Run 2_Data_Cleaning/data_cleaning.ipynb to regenerate cleaned_data.csv')
print('  3. Run 3_EDA/eda.ipynb for visualisations')


  RUN SUMMARY
  Files in data/      : 24
  Already processed   : 15
  Newly processed     : 1
  Total in registry   : 16
  Master rows         : 7,797
  Unique users        : 16

Next steps:
  1. Fill in demographics_template.csv and save as demographics.csv
  2. Run 2_Data_Cleaning/data_cleaning.ipynb to regenerate cleaned_data.csv
  3. Run 3_EDA/eda.ipynb for visualisations


## 🔎 Step 15 — (Optional) View Processed File Registry

Run this cell at any time to see which files have been processed, when, and which USER_N was assigned. This is your audit trail.

In [30]:
reg = load_registry()
if reg['processed_files']:
    df_reg = pd.DataFrame(reg['processed_files'])
    display(df_reg[['filename','processed_at','user_alias',
                     'confirmed_name','transactions_added']])
else:
    print('Registry is empty — no files have been processed yet.')


,filename,processed_at,user_alias,confirmed_name,transactions_added
0,Messages avec MobileMoney 2026-03-20 21-32-36 ...,2026-05-05 22:09:29,USER_1,Marie madeleine Tchayo,147
1,Messages avec MobileMoney 2026-03-22 12_27_25 ...,2026-05-05 22:09:56,USER_2,JACOB FAH,315
2,Messages avec OrangeMoney 2026-03-20 21-31-49 ...,2026-05-05 22:10:29,USER_3,TCHAYO,894
3,Messages avec OrangeMoney 2026-03-20 22_05_33 ...,2026-05-05 22:10:42,USER_4,Bedibiki bougong,147
4,Messages avec OrangeMoney 2026-03-21 11_16_17 ...,2026-05-05 22:11:00,USER_5,NDJANA MENGUE,1525
5,Messages avec OrangeMoney 2026-03-22 12_28_16 ...,2026-05-05 22:11:09,USER_6,FAH,862
6,Messages avec OrangeMoney 2026-03-23 014528 - ...,2026-05-05 22:11:15,USER_7,ABOLO,518
7,Messages avec OrangeMoney Brown 2026-03-23 083...,2026-05-05 22:11:29,USER_8,DJOUTSOP TAKOU,58
8,Messages with MobileMoney 2026-03-20 21_13_48 ...,2026-05-05 22:14:04,USER_9,JOEL XAVIER DEJON FAH,455
9,Messages with MobileMoney 2026-03-25 145028.xlsx,2026-05-05 22:17:21,USER_10,GENESIS MUDOH TEYIM,795
